# Stacked BiLSTM

ByteLevel BPE → Embedding 128 → 3-layer BiLSTM, hidden 256 per direction → Dropout → Linear 512 → 3.

## Setup



In [4]:
import os
import sys
import base64
import subprocess
from pathlib import Path

ENVIRONMENT = "colab" if "google.colab" in sys.modules else "kaggle"
root = Path("/content" if ENVIRONMENT == "colab" else "/kaggle/working")
repo_dir = Path(os.environ.get("UA_NER_REPO_DIR", root / "UA-Locations-NER-Deep-Learning"))

if not repo_dir.exists():
    if ENVIRONMENT == "colab":
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
               GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
    try:
        subprocess.run(["git", "clone", "-b", "stacked",
                        "https://github.com/dkalchenko01/UA-Locations-NER-Deep-Learning.git",
                        str(repo_dir)], env=env, check=True)
    finally:
        del token, auth, env

sys.path.insert(0, str(repo_dir))
print(repo_dir)



/content/UA-Locations-NER-Deep-Learning


In [5]:
%pip install -q -r "$repo_dir/requirements.txt"



In [6]:
import gc
import json
import platform
import uuid
from importlib.metadata import version

import pandas as pd
import torch
from tokenizers import Tokenizer

from src.data_io import (get_data_dir, load_manifest, load_data_splits,
                         save_json, save_predictions, file_sha256)
from src.dataset import prepare_ner_records, prepare_inference_records, make_loader, LABEL_TO_ID
from src.model import BiLSTMTagger
from src.metrics import competition_scores
from src.preprocessing import clean_locations_v3, serialize_spans
from src.training import set_seed, train_epoch, predict, save_checkpoint

import wandb
from time import perf_counter
from src.experiment_tracking import start_run, log_epoch, log_evaluation, finish_run


## W&B login


In [7]:
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"
os.environ["WANDB_DISABLE_CODE"] = "true"
os.environ["WANDB_CONSOLE"] = "off"

if ENVIRONMENT == "colab":
    from google.colab import userdata
    wandb.login(key=userdata.get("WANDB_API_KEY"), relogin=True)
else:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"), relogin=True)



/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: No netrc file found, creating one.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /root/.netrc
wandb: Currently logged in as: daldoshyna (daldoshyna-kyiv-school-of-economics) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


## Configuration



In [8]:
config = {
    "model": "bilstm", "architecture": "stacked", "embedding_dim": 128, "hidden_size": 256,
    "num_layers": 3, "dropout": 0.2, "seed": 17, "batch_size": 128,
    "learning_rate": 1e-3, "epochs": 5, "gradient_clip": 1.0,
    "max_padded_tokens": 16384, "prepare_batch_size": 2000,
    "sources": ["uk", "ru"], "optimizer": "AdamW",
    "tokenizer": "bpe_whitespace.json", "selection_metric": "uk_comp_f1",
    "label_to_id": LABEL_TO_ID, "max_sequence_length": None,
    "early_stopping": False, "environment": ENVIRONMENT,
    "validation_version": "v1", "metric_version": "kaggle_original_regex_v1",
    "postprocessing_version": "raw", "upload_checkpoint": False,
}
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
set_seed(config["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
run_id = "stacked_" + pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
run_dir = root / "outputs" / run_id
run_dir.mkdir(parents=True, exist_ok=False)
splits_dir = repo_dir / "data/splits"
tokenizer_path = repo_dir / "artifacts/tokenizers" / config["tokenizer"]

provenance = {
    "run_id": run_id, "seed": config["seed"], "python": platform.python_version(),
    "git_revision": subprocess.run(["git", "-C", str(repo_dir), "rev-parse", "HEAD"],
                                   capture_output=True, text=True).stdout.strip() or None,
    "git_status": subprocess.run(["git", "-C", str(repo_dir), "status", "--short"],
                                 capture_output=True, text=True).stdout.strip(),
    "packages": {name: version(name) for name in ["torch", "tokenizers", "numpy", "pandas", "pyarrow", "tqdm", "wandb"]},
    "device": str(device), "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
    "tokenizer_sha256": file_sha256(tokenizer_path),
    "manifest_sha256": file_sha256(splits_dir / "validation_manifest.parquet"),
    "source_sha256": {path.name: file_sha256(path) for path in sorted((repo_dir / "src").glob("*.py"))},
    "notebook_sha256": file_sha256(repo_dir / "notebooks/04_bilstm_stacked.ipynb"),
}
save_json(run_dir / "config.json", config)
save_json(run_dir / "provenance.json", provenance)
print(device, run_dir)



cuda /content/outputs/stacked_20261112T221414Z_334876aa


## Experiment tracking


In [9]:
os.environ["WANDB_DIR"] = str(run_dir)
run = start_run(
    model="bilstm", config=config, name=run_id,
    validation_version=config["validation_version"], metric_version=config["metric_version"],
    postprocessing_version=config["postprocessing_version"], repo_dir=repo_dir,
    split_manifest_path=splits_dir / "validation_manifest.parquet",
    tokenizer_path=tokenizer_path, provenance=provenance,
    tags=["stacked", f"layers-{config['num_layers']}", f"hidden-{config['hidden_size']}"]
)
save_json(run_dir / "wandb_run.json", {"id": run.id, "url": run.url,
                                      "path": f"{run.entity}/{run.project}/{run.id}"})
print(run.url)



https://app.forge.coreweave.com/daldoshyna-kyiv-school-of-economics/ua-location-ner/runs/7kn5bag9


## Data



In [10]:
tokenizer = Tokenizer.from_file(str(tokenizer_path))
tokenizer.no_padding()
tokenizer.no_truncation()
pad_id = tokenizer.token_to_id("[PAD]")
data_dir = get_data_dir(ENVIRONMENT)
manifest = load_manifest(splits_dir)
display(pd.crosstab(manifest["source"], manifest["split"]))
del manifest

train_records, val_records, alignment = [], [], []
for source in config["sources"]:
    source_splits = load_data_splits(ENVIRONMENT, splits_dir, sources=(source,))
    for split in ("train", "val"):
        frame = source_splits.pop(f"{source}_{split}")[["text", "loc_markers"]]
        records, skipped = prepare_ner_records(tokenizer, frame, source, split,
                                               batch_size=config["prepare_batch_size"])
        (train_records if split == "train" else val_records).extend(records)
        alignment.append({"source": source, "split": split, "total": len(frame),
                          "retained": len(records), "alignment_failures": len(skipped)})
        save_json(run_dir / f"{source}_{split}_skipped.json", skipped)
        del frame, records, skipped
        gc.collect()
    del source_splits
alignment = pd.DataFrame(alignment)
display(alignment)
alignment.to_csv(run_dir / "alignment.csv", index=False)



100%|██████████| 811M/811M [00:48<00:00, 17.7MB/s]


Extracting files...


split,train,val
source,,
ru,100000,49369
uk,805983,201498


Prepare uk/train:   0%|          | 0/403 [00:00<?, ?it/s]

Prepare uk/val:   0%|          | 0/101 [00:00<?, ?it/s]

Prepare ru/train:   0%|          | 0/50 [00:00<?, ?it/s]

Prepare ru/val:   0%|          | 0/25 [00:00<?, ?it/s]

,source,split,total,retained,alignment_failures
0,uk,train,805983,804367,1616
1,uk,val,201498,201498,375
2,ru,train,100000,99909,91
3,ru,val,49369,49369,38


In [11]:
loader_options = {
    "pad_id": pad_id, "batch_size": config["batch_size"], "seed": config["seed"],
    "max_padded_tokens": config["max_padded_tokens"], "pin_memory": device.type == "cuda"
}
train_loader = make_loader(train_records, shuffle=True, **loader_options)
val_loader = make_loader(val_records, **loader_options)



In [12]:
"train_records" in globals(), "train_loader" in globals()

(True, True)

## Model



In [13]:
model_config = {
    "vocab_size": tokenizer.get_vocab_size(), "pad_id": pad_id,
    "embedding_dim": config["embedding_dim"], "hidden_size": config["hidden_size"],
    "num_layers": config["num_layers"], "dropout": config["dropout"], "num_labels": len(LABEL_TO_ID)
}
model = BiLSTMTagger(**model_config).to(device)
optimizer_class = getattr(torch.optim, config["optimizer"])
optimizer = optimizer_class(model.parameters(), lr=config["learning_rate"])
save_json(run_dir / "model_config.json", model_config)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

run.config.update({"model_config": model_config})


BiLSTMTagger(
  (embedding): Embedding(32000, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=3, batch_first=True, dropout=0.2, bidirectional=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (classifier): Linear(in_features=512, out_features=3, bias=True)
)
Parameters: 8041987


## Training



In [14]:
history, best_score = [], -1
best_path = run_dir / "best.pt"
for epoch in range(1, config["epochs"] + 1):
    epoch_started = perf_counter()
    train_loader.batch_sampler.set_epoch(epoch)
    train_loss = train_epoch(model, train_loader, optimizer, device, config["gradient_clip"])
    predictions, val_losses = predict(model, val_loader, device)
    scores = competition_scores(val_records, predictions)
    clean_scores = competition_scores(val_records, predictions, cleaner=clean_locations_v3)
    history.append({"epoch": epoch, "train_loss": train_loss,
                    "epoch_seconds": perf_counter() - epoch_started,
                    **{f"{source}_val_loss": value for source, value in val_losses.items()},
                    **scores, **{f"clean_{key}": value for key, value in clean_scores.items()}})
    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    save_checkpoint(run_dir / f"epoch_{epoch:02d}.pt", model, optimizer, epoch, config, model_config, scores, provenance)
    if scores[config["selection_metric"]] > best_score:
        best_score = scores[config["selection_metric"]]
        save_checkpoint(best_path, model, optimizer, epoch, config, model_config, scores, provenance)
    log_epoch(run, epoch, history[-1], selection_metric=config["selection_metric"],
              epoch_seconds=history[-1]["epoch_seconds"],
              extra_metrics={f"val_{config['validation_version']}/clean_v3/{key}": value
                             for key, value in clean_scores.items()})
    print(history[-1])
    del predictions

display(pd.DataFrame(history))
optimizer.zero_grad(set_to_none=True)
del optimizer
gc.collect()


Train:   0%|          | 0/7106 [00:01<?, ?it/s]

Predict:   0%|          | 0/1971 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.03443547382984432, 'epoch_seconds': 168.209582262999, 'uk_val_loss': 0.019331747962464108, 'ru_val_loss': 0.025922959172094622, 'combined_val_loss': 0.02097238531389886, 'uk_comp_f1': 0.8663181, 'ru_comp_f1': 0.8586305, 'combined_comp_f1': 0.86438411, 'clean_uk_comp_f1': 0.86617196, 'clean_ru_comp_f1': 0.8586305, 'clean_combined_comp_f1': 0.86427459}


Train:   0%|          | 0/7102 [00:01<?, ?it/s]

Predict:   0%|          | 0/1971 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 0.016510757026245673, 'epoch_seconds': 166.78918608499953, 'uk_val_loss': 0.016654520088371278, 'ru_val_loss': 0.02155039987747676, 'combined_val_loss': 0.017873167616826252, 'uk_comp_f1': 0.88221487, 'ru_comp_f1': 0.88685499, 'combined_comp_f1': 0.88338538, 'clean_uk_comp_f1': 0.88209723, 'clean_ru_comp_f1': 0.88685499, 'clean_combined_comp_f1': 0.88329753}


... (7 outputs left out)

Predict:   0%|          | 0/1971 [00:00<?, ?it/s]

{'epoch': 5, 'train_loss': 0.007482869434152636, 'epoch_seconds': 170.40463163599998, 'uk_val_loss': 0.018208385334654256, 'ru_val_loss': 0.02169595377827144, 'combined_val_loss': 0.01907648603303929, 'uk_comp_f1': 0.89037319, 'ru_comp_f1': 0.90838996, 'combined_comp_f1': 0.89492933, 'clean_uk_comp_f1': 0.89000052, 'clean_ru_comp_f1': 0.90840975, 'clean_combined_comp_f1': 0.89465852}


,epoch,train_loss,epoch_seconds,uk_val_loss,ru_val_loss,combined_val_loss,uk_comp_f1,ru_comp_f1,combined_comp_f1,clean_uk_comp_f1,clean_ru_comp_f1,clean_combined_comp_f1
0,1,0.034435,168.209582,0.019332,0.025923,0.020972,0.866318,0.858630,0.864384,0.866172,0.858630,0.864275
1,2,0.016511,166.789186,0.016655,0.021550,0.017873,0.882215,0.886855,0.883385,0.882097,0.886855,0.883298
2,3,0.012362,167.729471,0.015951,0.020176,0.017003,0.886156,0.893281,0.887958,0.885862,0.893281,0.887739
3,4,0.009507,169.987567,0.016814,0.020650,0.017769,0.889135,0.907500,0.893758,0.888709,0.907520,0.893445
4,5,0.007483,170.404632,0.018208,0.021696,0.019076,0.890373,0.908390,0.894929,0.890001,0.908410,0.894659


18

## Select checkpoint


In [15]:
checkpoint_path = run_dir / "epoch_03.pt"
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
model = BiLSTMTagger(**checkpoint["model_config"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
print("Selected epoch:", checkpoint["epoch"])
summary = {"selected_epoch": checkpoint["epoch"], "checkpoint": checkpoint_path.name,
           "selection_scores": checkpoint["scores"]}
del checkpoint

predictions, val_losses = predict(model, val_loader, device)
summary.update(raw=competition_scores(val_records, predictions),
               clean_v3=competition_scores(val_records, predictions, cleaner=clean_locations_v3),
               validation_losses=val_losses)
save_json(run_dir / "summary.json", summary)
save_predictions(run_dir / "val_predictions_selected.jsonl", val_records, predictions)
print(summary)
del predictions

# log_evaluation(run, dataset=f"val_{config['validation_version']}", metrics=summary["raw"],
#                postprocessing_version="raw", kind="evaluation")
# log_evaluation(run, dataset=f"val_{config['validation_version']}", metrics=summary["clean_v3"],
#                postprocessing_version="clean_v3", kind="evaluation")



Selected epoch: 3


Predict:   0%|          | 0/1971 [00:00<?, ?it/s]

{'selected_epoch': 3, 'checkpoint': 'epoch_03.pt', 'selection_scores': {'uk_comp_f1': 0.88615603, 'ru_comp_f1': 0.89328063, 'combined_comp_f1': 0.88795755}, 'raw': {'uk_comp_f1': 0.88615603, 'ru_comp_f1': 0.89328063, 'combined_comp_f1': 0.88795755}, 'clean_v3': {'uk_comp_f1': 0.88586237, 'ru_comp_f1': 0.89328063, 'combined_comp_f1': 0.88773876}, 'validation_losses': {'uk': 0.015950904740233276, 'ru': 0.020175820445995096, 'combined': 0.01700254066769504}}


## Saving data for ensemble

In [22]:
import importlib
import src.decoding
import src.training

importlib.reload(src.decoding)
importlib.reload(src.training)

from src.training import predict
from src.export_predictions import export_predictions
from src.data_io import file_sha256, save_json

def export_for_ensemble(checkpoint_path, export_name):
    checkpoint_path = Path(checkpoint_path)
    export_dir = Path("/content/ensemble_exports") / export_name

    def predict_raw(model, loader):
        return predict(model, loader, device, decoding="viterbi_lines")

    export_predictions(
        output_dir=export_dir,
        checkpoint_path=checkpoint_path,
        model=model,
        predict_fn=predict_raw,
        val_records=val_records,
        val_loader=val_loader,
        test_records=test_records,
        test_loader=make_loader(test_records, **loader_options),
    )

    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=False
    )
    save_json(export_dir / "metadata.json", {
        "checkpoint": "checkpoint.pt",
        "checkpoint_original_name": checkpoint_path.name,
        "checkpoint_sha256": file_sha256(checkpoint_path),
        "epoch": checkpoint["epoch"],
        "model_config": checkpoint["model_config"],
        "training_config": checkpoint["config"],
        "decoding": "viterbi_lines",
        "postprocessing": "none",
        "tokenizer": "tokenizer.json",
        "tokenizer_sha256": file_sha256(tokenizer_path),
        "val_rows": len(val_records),
        "test_rows": len(test_records),
    })
    del checkpoint

    return export_dir

export_dir = export_for_ensemble(
    checkpoint_path,
    export_name=f"{run_dir.name}_{Path(checkpoint_path).stem}_viterbi_lines",
)

Checkpoint: /content/outputs/stacked_20261112T221414Z_334876aa/epoch_03.pt epoch: 3


Predict:   0%|          | 0/1971 [00:00<?, ?it/s]

Predict:   0%|          | 0/6 [00:00<?, ?it/s]

val.jsonl 250867 texts
test.jsonl 477 texts


In [23]:
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

def upload_ensemble_package(export_dir, checkpoint_path, tokenizer_path):
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    service = build("drive", "v3", credentials=credentials)
    parent_id = "1TXxyVIo1EtLJN-bXTp2HavZdzlgnDJ5j"

    files = {
        "val.jsonl": export_dir / "val.jsonl",
        "test.jsonl": export_dir / "test.jsonl",
        "metadata.json": export_dir / "metadata.json",
        "checkpoint.pt": Path(checkpoint_path),
        "tokenizer.json": Path(tokenizer_path),
    }

    for path in files.values():
        if not path.is_file():
            raise FileNotFoundError(path)

    def find_item(parent, name):
        escaped = name.replace(chr(92), chr(92) * 2).replace(
            "'", chr(92) + "'"
        )
        result = service.files().list(
            q=f"'{parent}' in parents and name = '{escaped}' and trashed = false",
            fields="files(id)",
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        return next(iter(result.get("files", [])), None)

    folder = find_item(parent_id, export_dir.name)
    if folder is None:
        folder = service.files().create(
            body={
                "name": export_dir.name,
                "mimeType": "application/vnd.google-apps.folder",
                "parents": [parent_id],
            },
            fields="id",
            supportsAllDrives=True,
        ).execute()

    for name, path in files.items():
        media = MediaFileUpload(str(path), resumable=True)
        existing = find_item(folder["id"], name)

        if existing:
            request = service.files().update(
                fileId=existing["id"],
                media_body=media,
                supportsAllDrives=True,
            )
        else:
            request = service.files().create(
                body={"name": name, "parents": [folder["id"]]},
                media_body=media,
                supportsAllDrives=True,
            )

        response = None
        while response is None:
            _, response = request.next_chunk()
        print("Uploaded:", name)

    print(f"https://drive.google.com/drive/folders/{folder['id']}")

upload_ensemble_package(export_dir, checkpoint_path, tokenizer_path)

Uploaded: val.jsonl
Uploaded: test.jsonl
Uploaded: metadata.json
Uploaded: checkpoint.pt
Uploaded: tokenizer.json
https://drive.google.com/drive/folders/1_crsEuABSHPrXbSAXsYQaKzR_0ErZ52_


## Test inference



In [16]:
test = pd.read_csv(data_dir / "test.csv", usecols=["text_id", "text"])
test_records = prepare_inference_records(tokenizer, test)
test_predictions, _ = predict(model, make_loader(test_records, **loader_options), device)
save_predictions(run_dir / "test_predictions.jsonl", test_records, test_predictions)

for filename, cleaner in [("submission.csv", list), ("submission_clean_v3.csv", clean_locations_v3)]:
    submission = test[["text_id"]].copy()
    submission["locations"] = submission["text_id"].map(
        lambda text_id: json.dumps(cleaner(test_predictions[text_id]["locations"]), ensure_ascii=False))
    submission.to_csv(run_dir / filename, index=False)
    print(run_dir / filename)
display(submission.head())

Predict:   0%|          | 0/6 [00:00<?, ?it/s]

/content/outputs/stacked_20261112T221414Z_334876aa/submission.csv
/content/outputs/stacked_20261112T221414Z_334876aa/submission_clean_v3.csv


,text_id,locations
0,0,"[""Київ""]"
1,1,"[""Києві"", ""Шулявського шляхопроводу"", ""Шулявсь..."
2,2,"[""Гоголеве"", ""Миргородського району"", ""Полтавс..."
3,3,[]
4,4,"[""проспекті Академіка Глушкова, 9""]"


## Post-processing

In [18]:
import importlib
import src.decoding
import src.training
import src.postprocessing

importlib.reload(src.decoding)
importlib.reload(src.training)
importlib.reload(src.postprocessing)

from src.training import predict
from src.postprocessing import postprocess

clean_options = {"expand": True, "trim": True, "empty": True}

viterbi_predictions, _ = predict(
    model,
    make_loader(test_records, **loader_options),
    device,
    decoding="viterbi_lines",
)

save_predictions(
    run_dir / "test_predictions_viterbi_lines.jsonl",
    test_records,
    viterbi_predictions,
)

submission = test[["text_id"]].copy()
submission["locations"] = [
    json.dumps(
        postprocess(
            row.text,
            viterbi_predictions[row.text_id]["spans"],
            **clean_options,
        ),
        ensure_ascii=False,
    )
    for row in test.itertuples(index=False)
]

path = run_dir / "submission_viterbi_lines_clean_v5.csv"
submission.to_csv(path, index=False)

save_json(run_dir / "viterbi_lines_config.json", {
    "checkpoint": str(checkpoint_path),
    "decoding": "viterbi_lines",
    "postprocessing": clean_options,
})

print(path)
display(submission.head())

Predict:   0%|          | 0/6 [00:00<?, ?it/s]

/content/outputs/stacked_20261112T221414Z_334876aa/submission_viterbi_lines_clean_v5.csv


,text_id,locations
0,0,"[""Київ""]"
1,1,"[""Києві"", ""Шулявського шляхопроводу"", ""Шулявсь..."
2,2,"[""Гоголеве"", ""Миргородського району"", ""Полтавс..."
3,3,[]
4,4,"[""проспекті Академіка Глушкова, 9""]"


In [19]:
changes = []

for row in test.itertuples(index=False):
    before = postprocess(
        row.text,
        test_predictions[row.text_id]["spans"],
        **clean_options,
    )
    after = postprocess(
        row.text,
        viterbi_predictions[row.text_id]["spans"],
        **clean_options,
    )

    if before != after:
        changes.append({
            "text_id": row.text_id,
            "text": row.text,
            "before": before,
            "after": after,
        })

for change in changes:
    print("text_id:", change["text_id"])
    print(change["text"])
    print("ARGMAX: ", json.dumps(change["before"], ensure_ascii=False))
    print("VITERBI_LINES:", json.dumps(change["after"], ensure_ascii=False))
    print()

pd.DataFrame(changes).to_json(
    run_dir / "viterbi_lines_changes.jsonl",
    orient="records", lines=True, force_ascii=False,
)

print("Changed messages:", len(changes))

text_id: 9
⛔️ Відзавтра у Солом’янському районі столиці на кілька місяців обмежать рух вулицею

Із 30 серпня на вулиці Новій у Солом’янському районі обмежать рух через ремонт.
ARGMAX:  ["Солом’янському районі столиці", "вулицею\n\nІз 30", "вулиці Новій", "Солом’янському районі"]
VITERBI_LINES: ["Солом’янському районі столиці", "вулицею", "Із 30", "вулиці Новій", "Солом’янському районі"]

text_id: 16
🕯У Києві відбувається церемонія прощання із загиблим військовим пілотом "Джусом"

Прощання відбувається у Патріаршому соборі Воскресіння Христового УГКЦ.

25 серпня на Житомирщині зіткнулися навчально-бойові літаки Л-39, троє пілотів загинули. Військовий пілот Андрій Пільщиков від початку повномасштабного вторгнення РФ здійснив десятки бойових вильотів.
ㅤ
ARGMAX:  ["Києві", "ар", "Житомирщині", "РФ"]
VITERBI_LINES: ["Києві", "Патріаршому", "Житомирщині", "РФ"]

text_id: 20
Оболонські лагуни.

Життя🫶Оболонь
ARGMAX:  ["нь"]
VITERBI_LINES: ["Оболонь"]

text_id: 37
☀️Цього тижня на киян чекає с

## Test chunking

In [20]:
# import re

# chunk_chars = 200
# test = pd.read_csv(data_dir / "test.csv", usecols=["text_id", "text"])
# chunks = []

# for row in test.itertuples(index=False):
#     for line in re.finditer(r"[^\r\n]+", row.text):
#         start, end = line.span()
#         while start < end:
#             stop = min(start + chunk_chars, end)
#             if stop < end:
#                 space = row.text.rfind(" ", start, stop)
#                 if space > start:
#                     stop = space
#             text = row.text[start:stop]
#             if text.strip():
#                 chunks.append({
#                     "text_id": len(chunks), "text": text,
#                     "original_id": row.text_id, "start": start,
#                 })
#             start = stop

# chunks = pd.DataFrame(chunks)
# chunk_records = prepare_inference_records(tokenizer, chunks)
# chunk_predictions, _ = predict(
#     model, make_loader(chunk_records, **loader_options), device
# )

In [21]:
# chunked_predictions = {
#     text_id: {"spans": [], "locations": []}
#     for text_id in test["text_id"]
# }

# for row in chunks.itertuples(index=False):
#     chunked_predictions[row.original_id]["spans"].extend(
#         (a + row.start, b + row.start)
#         for a, b in chunk_predictions[row.text_id]["spans"]
#     )

# for row in test.itertuples(index=False):
#     prediction = chunked_predictions[row.text_id]
#     prediction["spans"] = sorted(set(prediction["spans"]))
#     prediction["locations"] = [
#         row.text[a:b] for a, b in prediction["spans"]
#     ]

# chunk_dir = run_dir / f"chunked_{checkpoint_path.stem}_{chunk_chars}"
# chunk_dir.mkdir(exist_ok=True)
# save_json(chunk_dir / "config.json", {
#     "checkpoint": str(checkpoint_path), "chunk_chars": chunk_chars,
#     "split": "lines_then_spaces",
# })

# for filename, cleaner in [
#     ("submission_chunked.csv", list),
#     ("submission_chunked_clean_v3.csv", clean_locations_v3),
# ]:
#     submission = test[["text_id"]].copy()
#     submission["locations"] = submission["text_id"].map(
#         lambda text_id: json.dumps(
#             cleaner(chunked_predictions[text_id]["locations"]),
#             ensure_ascii=False,
#         )
#     )
#     submission.to_csv(chunk_dir / filename, index=False)
#     print(chunk_dir / filename)

# display(submission.head())